In [ ]:
"""
Step 2 - Data Ingestion (Raw CSV -> Bronze Parquet)
Customer Health & Sentiment Scoring Multi-Agent AI Solution

What this script does
  1. Reads the 4 source CSVs with an EXPLICIT schema taken from the data dictionary
     (no type guessing, so IDs like C0000001 stay as strings and nullable columns stay nullable).
  2. Captures malformed rows instead of failing (DuckDB rejects tables) -> logs/rejects_<table>.csv.
  3. Adds lineage columns: _source_file, _ingested_at.
  4. Writes each file to Parquet (ZSTD) in data/bronze/.
  5. Reconciles row counts and distinct customers against the known volumes (WARN, not fail).
  6. Registers Bronze views in the shared DuckDB database for the next steps.

What it deliberately does NOT do (that is Step 3b - Validate & clean):
  range checks, allowed-value checks, dedupe, FK checks, null handling.

Run:  python step2_data_ingestion.py            (or: python run_pipeline.py --only 2)
      STRICT_VOLUMES=1 python step2_data_ingestion.py   -> volume mismatches fail the step

Change log (see decisions.md): D01 renamed from 1.py and moved onto common.py paths/connection (PIPELINE_ROOT works),
D02 csat/nps read as DOUBLE so a bad rating no longer rejects the whole interaction row, D03 main() returns bool.
"""

## 1. Imports and configuration

In [ ]:
from __future__ import annotations

import json
import os
import time
from datetime import datetime

import duckdb

from common import (BRONZE_DIR, LOG_DIR, RAW_DIR, finish, get_connection, log, sql_path)

# Set these only if DuckDB fails to auto-detect your date formats,
# e.g. TIMESTAMP_FORMAT = "%Y-%m-%d %H:%M:%S", DATE_FORMAT = "%Y-%m-%d"
TIMESTAMP_FORMAT: str | None = None
DATE_FORMAT: str | None = None

DISTINCT_TOLERANCE = 0.05                          # +/- 5 % for "~" distinct-customer figures
STRICT_VOLUMES = os.getenv("STRICT_VOLUMES", "0") == "1"

## 2. Source definitions (schema and known volumes from the data dictionary)

In [ ]:
SOURCES: dict[str, dict] = {
    "customers": {
        "file": "customers.csv", "pk": "customer_id",
        "expected_rows": 1_000_000, "expected_distinct_customers": 1_000_000,
        "columns": {
            "customer_id": "VARCHAR", "signup_date": "DATE", "age": "INTEGER", "gender": "VARCHAR",
            "region": "VARCHAR", "country": "VARCHAR", "customer_segment": "VARCHAR",
            "acquisition_channel": "VARCHAR", "income_band": "VARCHAR", "preferred_channel": "VARCHAR",
            "loyalty_tier": "VARCHAR", "tenure_months": "INTEGER", "marketing_consent": "VARCHAR",
        },
    },
    "interactions": {
        "file": "interactions.csv", "pk": "interaction_id",
        "expected_rows": 1_000_000, "expected_distinct_customers": 526_000,
        "columns": {
            "interaction_id": "VARCHAR", "customer_id": "VARCHAR", "interaction_ts": "TIMESTAMP",
            "month": "VARCHAR", "channel": "VARCHAR", "interaction_type": "VARCHAR", "agent_id": "VARCHAR",
            "duration_sec": "INTEGER", "resolution_status": "VARCHAR", "first_contact_resolution": "VARCHAR",
            "csat_score": "DOUBLE",         # nullable (~45%); DOUBLE so 3b can NULL a non-integer rating (D02)
            "nps_score": "DOUBLE",          # nullable (~12%); same reason
            "interaction_text": "VARCHAR",
            "sentiment_label": "VARCHAR",   # ground truth - validation only
            "sentiment_score_gt": "DOUBLE", # ground truth - validation only
        },
    },
    "transactions": {
        "file": "transactions.csv", "pk": "transaction_id",
        "expected_rows": 1_500_000, "expected_distinct_customers": 560_000,
        "columns": {
            "transaction_id": "VARCHAR", "customer_id": "VARCHAR", "transaction_ts": "TIMESTAMP",
            "month": "VARCHAR", "transaction_type": "VARCHAR", "product_category": "VARCHAR",
            "amount": "DOUBLE",             # local currency, negative for refunds
            "currency": "VARCHAR", "discount_amount": "DOUBLE", "payment_method": "VARCHAR",
            "merchant_channel": "VARCHAR", "transaction_status": "VARCHAR", "is_international": "VARCHAR",
        },
    },
    "engagements": {
        "file": "engagements.csv", "pk": "engagement_id",
        "expected_rows": 1_500_000, "expected_distinct_customers": 590_000,
        "columns": {
            "engagement_id": "VARCHAR", "customer_id": "VARCHAR", "event_ts": "TIMESTAMP", "month": "VARCHAR",
            "engagement_channel": "VARCHAR", "event_type": "VARCHAR", "session_duration_sec": "INTEGER",
            "pages_viewed": "INTEGER", "campaign_id": "VARCHAR", "device_type": "VARCHAR",
            "app_version": "VARCHAR", "is_converted": "VARCHAR",
        },
    },
}

## 3. Helpers

In [ ]:
def check_inputs() -> None:
    missing = [s["file"] for s in SOURCES.values() if not (RAW_DIR / s["file"]).exists()]
    if missing:
        raise FileNotFoundError(f"Missing input files in {RAW_DIR}: {missing}")


def check_header(con: duckdb.DuckDBPyConnection, name: str, cfg: dict) -> None:
    """Fail fast if CSV column names/order differ from the data dictionary."""
    path = sql_path(RAW_DIR / cfg["file"])
    actual = [r[0] for r in con.execute(
        f"DESCRIBE SELECT * FROM read_csv('{path}', header = true, all_varchar = true)").fetchall()]
    expected = list(cfg["columns"])
    if actual != expected:
        raise ValueError(f"{name}: header mismatch\n  expected: {expected}\n  actual:   {actual}")


def build_read_csv(name: str, cfg: dict) -> str:
    """read_csv(...) expression with explicit schema and reject capture."""
    path = sql_path(RAW_DIR / cfg["file"])
    cols = ", ".join(f"'{c}': '{t}'" for c, t in cfg["columns"].items())
    opts = [f"'{path}'", "header = true", "delim = ','", "quote = '\"'", "escape = '\"'",
            f"columns = {{{cols}}}", "nullstr = ''", "store_rejects = true",
            f"rejects_table = 'rejects_{name}'", f"rejects_scan = 'rejects_scan_{name}'"]
    if TIMESTAMP_FORMAT:
        opts.append(f"timestampformat = '{TIMESTAMP_FORMAT}'")
    if DATE_FORMAT:
        opts.append(f"dateformat = '{DATE_FORMAT}'")
    return "read_csv(" + ", ".join(opts) + ")"

## 4. Ingestion

In [ ]:
def ingest_file(con: duckdb.DuckDBPyConnection, name: str, cfg: dict) -> dict:
    start = time.time()
    out_file = BRONZE_DIR / f"{name}.parquet"
    log(f"Ingesting {cfg['file']} -> {out_file.name}")
    check_header(con, name, cfg)
    # Load into a DuckDB table first so rejects are captured in the same connection
    con.execute(f"""
        CREATE OR REPLACE TABLE bronze_{name} AS
        SELECT *, '{cfg['file']}' AS _source_file, current_timestamp AS _ingested_at
        FROM {build_read_csv(name, cfg)}""")
    con.execute(f"COPY bronze_{name} TO '{sql_path(out_file)}' "
                f"(FORMAT parquet, COMPRESSION zstd, ROW_GROUP_SIZE 122880)")
    rows, distinct_pk, distinct_cust = con.execute(
        f"SELECT count(*), count(DISTINCT {cfg['pk']}), count(DISTINCT customer_id) FROM bronze_{name}").fetchone()
    rejected = con.execute(f"SELECT count(*) FROM rejects_{name}").fetchone()[0]
    if rejected:
        con.execute(f"COPY (SELECT * FROM rejects_{name}) TO "
                    f"'{sql_path(LOG_DIR / f'rejects_{name}.csv')}' (HEADER, DELIMITER ',')")
    exp_rows, exp_cust = cfg["expected_rows"], cfg["expected_distinct_customers"]
    result = {
        "table": name, "rows_loaded": rows, "rows_rejected": rejected, "rows_expected": exp_rows,
        "row_count_ok": rows + rejected == exp_rows, "pk_unique": distinct_pk == rows,
        "distinct_customers": distinct_cust, "distinct_customers_expected": exp_cust,
        "distinct_customers_ok": abs(distinct_cust - exp_cust) <= exp_cust * DISTINCT_TOLERANCE,
        "parquet_mb": round(out_file.stat().st_size / 1_048_576, 1), "seconds": round(time.time() - start, 1),
    }
    log(f"  {rows:,} loaded | {rejected:,} rejected | {distinct_cust:,} customers | "
        f"{result['parquet_mb']} MB | {result['seconds']}s")
    return result


def create_bronze_views(con: duckdb.DuckDBPyConnection) -> None:
    """Point stable views at the Parquet files and drop the staging tables so the .duckdb file stays small."""
    for name in SOURCES:
        con.execute(f"CREATE OR REPLACE VIEW v_bronze_{name} AS "
                    f"SELECT * FROM read_parquet('{sql_path(BRONZE_DIR / f'{name}.parquet')}')")
        con.execute(f"DROP TABLE IF EXISTS bronze_{name}")


def print_summary(results: list[dict]) -> bool:
    """Volume / PK / reject checks are WARNINGS (Step 3b deduplicates and quarantines); STRICT_VOLUMES=1 fails."""
    print("\n" + "=" * 96)
    print(f"{'table':<14}{'loaded':>12}{'rejected':>10}{'expected':>12}{'rows':>7}{'PK':>6}"
          f"{'customers':>12}{'cust':>7}{'MB':>8}")
    print("-" * 96)
    ok_ = lambda b: "OK" if b else "WARN"
    clean = True
    for r in results:
        print(f"{r['table']:<14}{r['rows_loaded']:>12,}{r['rows_rejected']:>10,}{r['rows_expected']:>12,}"
              f"{ok_(r['row_count_ok']):>7}{ok_(r['pk_unique']):>6}{r['distinct_customers']:>12,}"
              f"{ok_(r['distinct_customers_ok']):>7}{r['parquet_mb']:>8}")
        clean &= r["row_count_ok"] and r["pk_unique"] and r["distinct_customers_ok"] and r["rows_rejected"] == 0
    print("=" * 96)
    print("INGESTION STATUS:", "PASSED" if clean else "PASSED WITH WARNINGS (see logs/)")
    return clean

## 5. Main

In [ ]:
def main() -> bool:
    BRONZE_DIR.mkdir(parents=True, exist_ok=True)
    LOG_DIR.mkdir(parents=True, exist_ok=True)
    check_inputs()
    con = get_connection()
    log(f"DuckDB {duckdb.__version__} | raw = {RAW_DIR}")
    try:
        results = [ingest_file(con, name, cfg) for name, cfg in SOURCES.items()]
        create_bronze_views(con)
    finally:
        con.close()
    clean = print_summary(results)
    audit = {"run_at": datetime.now().isoformat(timespec="seconds"), "duckdb_version": duckdb.__version__,
             "status": "PASSED" if clean else "WARNINGS", "results": results}
    audit_file = LOG_DIR / f"ingestion_audit_{datetime.now():%Y%m%d_%H%M%S}.json"
    audit_file.write_text(json.dumps(audit, indent=2, default=str))
    log(f"Audit log written to {audit_file}")
    return clean or not STRICT_VOLUMES

In [ ]:
if __name__ == "__main__":
    finish(main())